# **Propensity score: сопоставление, взвешивание и дважды робастная оценка**

Практика к модулю [«Каузальность и линейные модели»](https://ai-interpretability.school), урок «Propensity score: сопоставление и взвешивание».

В уроке методы на propensity score были формулами и таблицей из шести сотрудниц. Здесь мы посчитаем их на одних данных и сравним с истинным эффектом. Проверим четыре утверждения урока:

1. сопоставление оценивает эффект для группы воздействия (ATT), а взвешивание — для всех (ATE), и при неоднородном эффекте это разные числа;
2. оценка Гаека устойчивее оценки Хорвица–Томпсона;
3. проверка баланса находит неверную модель propensity score, а высокий ROC-AUC этой модели ничего не доказывает;
4. дважды робастная оценка (AIPW) не смещена, если верна хотя бы одна из двух моделей, и смещена, если неверны обе.

План тетради:

1. данные с известными потенциальными исходами;
2. модель propensity score;
3. сопоставление (PSM);
4. взвешивание: Хорвиц–Томпсон и Гаек;
5. проверка баланса;
6. AIPW при верных и неверных моделях;
7. где ломается: нарушение позитивности, обрезка и trimming;
8. те же оценки в DoWhy.

Тетрадь идет на процессоре, прогон занимает меньше минуты.

In [ ]:
%pip install -q dowhy==0.14

In [ ]:
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')
SEED = 0
pd.set_option('display.width', 140)

## 1. Данные

Компания отправляет сотрудниц на курс повышения квалификации ($T=1$) и через год смотрит на прибавку к зарплате $Y$ в тысячах. Признаки:

- $X_1$ — стаж (стандартизован);
- $X_2$ — оценка за прошлый год (стандартизована). На курс чаще отправляют сотрудниц с крайними оценками, и очень низкими, и очень высокими, и у них же выше прибавка, поэтому в обеих моделях стоит $Q = X_2^2 - 1$;
- $X_3$ — работает ли сотрудница в головном офисе.

$$\operatorname{logit} e(x) = -0.3 + 0.7X_1 + 0.4Q + 0.4X_3,$$
$$Y_0 = 20 + 2X_1 + 3Q + 1.5X_3 + \varepsilon,\quad \varepsilon\sim\mathcal N(0, 2^2),\qquad Y_1 = Y_0 + \tau(x),\quad \tau(x) = 2 + X_1.$$

Эффект неоднороден: опытным курс дает больше. Опытных на курс отправляют чаще, поэтому ATT больше ATE. Оба потенциальных исхода сгенерированы, и истинные ATE и ATT на этой выборке посчитаны прямо по ним.

In [ ]:
def simulate(n=10000, k=1.0, seed=SEED):
    rng = np.random.default_rng(seed)
    X1 = rng.normal(size=n); X2 = rng.normal(size=n); X3 = rng.binomial(1, 0.4, n)
    Q = X2 ** 2 - 1
    e = 1 / (1 + np.exp(-k * (-0.3 + 0.7 * X1 + 0.4 * Q + 0.4 * X3)))   # k > 1 делает группы менее похожими
    T = rng.binomial(1, e)
    Y0 = 20 + 2 * X1 + 3 * Q + 1.5 * X3 + rng.normal(scale=2, size=n)
    Y1 = Y0 + 2 + X1
    return pd.DataFrame(dict(X1=X1, X2=X2, X3=X3, T=T, Y=np.where(T == 1, Y1, Y0), Y0=Y0, Y1=Y1, e_true=e))

d = simulate()
t = d['T'].values == 1
true_ate = (d.Y1 - d.Y0).mean()
true_att = (d.Y1 - d.Y0)[t].mean()
naive = d.Y[t].mean() - d.Y[~t].mean()
print(f'объектов {len(d)}, на курсе {t.sum()} ({t.mean():.3f})')
print(f'истинный ATE {true_ate:.3f} | истинный ATT {true_att:.3f} | разность средних {naive:.3f}')

## 2. Модель propensity score

Две логистические регрессии: верная, со слагаемым $X_2^2$, и неверная, где $X_2$ входит линейно, как обычно и пишут, не зная, что важны крайние оценки. Колонки `Y0`, `Y1` и `e_true` в моделях не участвуют: в настоящих данных их нет.

In [ ]:
PS_FORMULA = {'верная': 'T ~ X1 + X3 + I(X2**2)', 'неверная': 'T ~ X1 + X2 + X3'}

def propensity(data, formula):
    return smf.logit(formula, data=data).fit(disp=0).predict(data).values

e_hat = {name: propensity(d, f) for name, f in PS_FORMULA.items()}
for name, e in e_hat.items():
    print(f'{name:9s} ROC-AUC {roc_auc_score(d["T"], e):.3f} | min e {e.min():.3f} | max e {e.max():.3f}')

## 3. Сопоставление

Для каждой сотрудницы с курса ищем контрольную с ближайшим логитом $\hat e$, с возвращением, и отбрасываем пары дальше калипера — 0.2 стандартного отклонения логита ([Austin, 2011](https://doi.org/10.1002/pst.433)). Оценка — средняя разность исходов в парах:

$$\widehat{ATT} = \frac{1}{N_1}\sum_{i:\,T_i=1}\big(Y_i - Y_{j(i)}\big).$$

In [ ]:
def psm_att(data, e, caliper_sd=0.2):
    t = data['T'].values == 1
    logit = np.log(e / (1 - e))
    nn = NearestNeighbors(n_neighbors=1).fit(logit[~t].reshape(-1, 1))
    dist, idx = nn.kneighbors(logit[t].reshape(-1, 1))
    keep = dist[:, 0] <= caliper_sd * logit.std()
    y = data['Y'].values
    return np.mean(y[t][keep] - y[~t][idx[keep, 0]]), int((~keep).sum())

att_psm, dropped = psm_att(d, e_hat['верная'])
print(f'PSM: ATT {att_psm:.3f} (истинный ATT {true_att:.3f}, истинный ATE {true_ate:.3f})')
print(f'без пары внутри калипера: {dropped} из {t.sum()}')

**Задание 1. Чему равна оценка ATT сопоставлением с верной моделью propensity score (три знака после точки)? Сколько сотрудниц с курса осталось без пары? К какому из двух истинных эффектов оценка ближе и почему?**

## 4. Взвешивание

Оценка Хорвица–Томпсона делит взвешенные суммы на $n$, оценка Гаека — на суммы весов своей группы:

$$\widehat{ATE}_{HT} = \frac1n\sum_i\frac{T_iY_i}{\hat e(x_i)} - \frac1n\sum_i\frac{(1-T_i)Y_i}{1-\hat e(x_i)},\qquad
\widehat{ATE}_{H} = \frac{\sum_i T_iY_i/\hat e(x_i)}{\sum_i T_i/\hat e(x_i)} - \frac{\sum_i (1-T_i)Y_i/(1-\hat e(x_i))}{\sum_i (1-T_i)/(1-\hat e(x_i))}.$$

In [ ]:
def ipw(data, e):
    T, Y = data['T'].values, data['Y'].values
    ht = np.mean(T * Y / e) - np.mean((1 - T) * Y / (1 - e))
    hajek = np.sum(T * Y / e) / np.sum(T / e) - np.sum((1 - T) * Y / (1 - e)) / np.sum((1 - T) / (1 - e))
    return ht, hajek

for name, e in e_hat.items():
    ht, hajek = ipw(d, e)
    print(f'модель {name:9s}: Хорвиц–Томпсон {ht:.3f} | Гаек {hajek:.3f} | истинный ATE {true_ate:.3f}')
w = np.where(t, 1 / e_hat['верная'], 1 / (1 - e_hat['верная']))
print(f'наибольший вес {w.max():.1f}, доля суммы весов у 1% объектов с крупнейшими весами {np.sort(w)[-100:].sum() / w.sum():.3f}')

**Задание 2. Чему равна оценка Гаека с верной моделью propensity score (три знака после точки)? Какая из двух оценок, Хорвица–Томпсона или Гаека, ближе к истинному ATE, и что дает неверная модель?**

## 5. Проверка баланса

Стандартизованная разность средних признака между группами: до взвешивания — простая, после — со взвешенными средними и дисперсиями. Порог 0.1 — распространенное эмпирическое правило ([Austin, 2011](https://pmc.ncbi.nlm.nih.gov/articles/PMC3144483/)). Проверяем не только $X_1, X_2, X_3$, но и $X_2^2$: баланс смотрят и по функциям признаков, которые могут входить в исход.

In [ ]:
def smd(x, T, w=None):
    w = np.ones_like(x, dtype=float) if w is None else w
    m1, m0 = np.average(x[T == 1], weights=w[T == 1]), np.average(x[T == 0], weights=w[T == 0])
    v1 = np.average((x[T == 1] - m1) ** 2, weights=w[T == 1])
    v0 = np.average((x[T == 0] - m0) ** 2, weights=w[T == 0])
    return (m1 - m0) / np.sqrt((v1 + v0) / 2)

T = d['T'].values
features = {'X1': d.X1.values, 'X2': d.X2.values, 'X3': d.X3.values.astype(float), 'X2^2': d.X2.values ** 2}
balance = pd.DataFrame({'до взвешивания': {k: smd(v, T) for k, v in features.items()}})
for name, e in e_hat.items():
    w = np.where(T == 1, 1 / e, 1 / (1 - e))
    balance[f'после, модель {name}'] = {k: smd(v, T, w) for k, v in features.items()}
balance.round(3)

Неверная модель выравнивает те признаки, которые в нее вошли, и оставляет разницу по $X_2^2$. ROC-AUC тут не помогает: у верной модели он выше лишь на несколько сотых, а на вопрос, уравнены ли группы, отвечает только таблица баланса.

## 6. Дважды робастная оценка

AIPW ([Robins, Rotnitzky и Zhao, 1994](https://doi.org/10.1080/01621459.1994.10476818)) берет прогноз модели исхода и поправляет его взвешенными остатками:

$$\widehat{ATE}_{AIPW} = \frac1n\sum_i\Big[\hat\mu_1(x_i) - \hat\mu_0(x_i) + \frac{T_i\big(Y_i - \hat\mu_1(x_i)\big)}{\hat e(x_i)} - \frac{(1-T_i)\big(Y_i - \hat\mu_0(x_i)\big)}{1 - \hat e(x_i)}\Big],$$

где $\hat\mu_1$ и $\hat\mu_0$ — модели исхода, обученные отдельно на группе воздействия и на контрольной. Модель исхода тоже бывает верной (со слагаемым $X_2^2$) и неверной. Переберем все четыре сочетания; рядом — оценка одной моделью исхода, $\frac1n\sum_i(\hat\mu_1(x_i) - \hat\mu_0(x_i))$.

In [ ]:
OUT_FORMULA = {'верная': 'Y ~ X1 + X3 + I(X2**2)', 'неверная': 'Y ~ X1 + X2 + X3'}

def outcome_models(data, formula):
    mu1 = smf.ols(formula, data=data[data['T'] == 1]).fit().predict(data).values
    mu0 = smf.ols(formula, data=data[data['T'] == 0]).fit().predict(data).values
    return mu1, mu0

def aipw(data, e, mu1, mu0):
    T, Y = data['T'].values, data['Y'].values
    return np.mean(mu1 - mu0 + T * (Y - mu1) / e - (1 - T) * (Y - mu0) / (1 - e))

rows = []
for out_name, f in OUT_FORMULA.items():
    mu1, mu0 = outcome_models(d, f)
    for ps_name, e in e_hat.items():
        rows.append(dict(модель_исхода=out_name, модель_ps=ps_name,
                         только_модель_исхода=round(np.mean(mu1 - mu0), 3), AIPW=round(aipw(d, e, mu1, mu0), 3)))
dr = pd.DataFrame(rows)
print(f'истинный ATE {true_ate:.3f}')
dr

**Задание 3. Чему равна оценка AIPW, когда модель исхода неверная, а модель propensity score верная (три знака после точки)? В каком из четырех сочетаний AIPW смещена, и что в этом сочетании дает одна модель исхода?**

Модели здесь обучены на тех же объектах, на которых считается оценка. С гибкими моделями (лес, бустинг) так делать нельзя: их переобучение попадает в оценку, поэтому прогнозы для каждого объекта берут из моделей, которые его не видели, — cross-fitting, как в DML из урока «IV и DML».

## 7. Где ломается: позитивность

Увеличим коэффициенты модели назначения в $k=2.5$ раза: курс теперь почти однозначно определяется признаками, и группы перекрываются слабо. Истинный эффект остается тем же. Два способа сладить с крайними весами из урока: обрезка — ограничить $\hat e$ отрезком $[0.05, 0.95]$, оставив всех; trimming — выбросить объекты, у которых $\hat e$ вне этого отрезка ([Crump и др., 2009](https://doi.org/10.1093/biomet/asn055)). Второй способ меняет совокупность, для которой оценен эффект, поэтому рядом — истинный ATE оставшихся.

In [ ]:
d_bad = simulate(k=2.5)
T_bad = d_bad['T'].values
e_bad = propensity(d_bad, PS_FORMULA['верная'])
w_bad = np.where(T_bad == 1, 1 / e_bad, 1 / (1 - e_bad))
ess = w_bad.sum() ** 2 / (w_bad ** 2).sum()
mu1_b, mu0_b = outcome_models(d_bad, OUT_FORMULA['верная'])
true_bad = (d_bad.Y1 - d_bad.Y0).mean()
print(f'ROC-AUC {roc_auc_score(T_bad, e_bad):.3f} | min e {e_bad.min():.4f} | max e {e_bad.max():.4f} | наибольший вес {w_bad.max():.0f} | эффективный размер выборки {ess:.0f} из {len(d_bad)}')

e_clip = np.clip(e_bad, 0.05, 0.95)
keep = (e_bad > 0.05) & (e_bad < 0.95)
rows = [
    dict(способ='без поправки', Гаек=ipw(d_bad, e_bad)[1], AIPW=aipw(d_bad, e_bad, mu1_b, mu0_b), истинный=true_bad),
    dict(способ='обрезка [0.05, 0.95]', Гаек=ipw(d_bad, e_clip)[1], AIPW=aipw(d_bad, e_clip, mu1_b, mu0_b), истинный=true_bad),
    dict(способ=f'trimming, осталось {keep.mean():.3f}', Гаек=ipw(d_bad[keep], e_bad[keep])[1],
         AIPW=aipw(d_bad[keep], e_bad[keep], mu1_b[keep], mu0_b[keep]), истинный=(d_bad.Y1 - d_bad.Y0)[keep].mean()),
]
pd.DataFrame(rows).round(3)

Эффективный размер выборки $\big(\sum_i w_i\big)^2/\sum_i w_i^2$ показывает, скольким объектам с равными весами равносильна взвешенная выборка. Обрезка смещает оценку Гаека: объекты с крайними $\hat e$ получают меньший вес, чем им положен, и смешение по ним возвращается. AIPW держится лучше, потому что поправка весами у нее — только поправка к модели исхода. Trimming отвечает на другой вопрос — об эффекте для объектов, у которых есть с кем сравнивать, — и с истинным эффектом надо сравнивать истинный эффект оставшихся.

## 8. Те же оценки в DoWhy

DoWhy строит propensity score логистической регрессией по переданным общим причинам, поэтому $X_2^2$ передаем отдельной колонкой. Сопоставление в DoWhy — ближайший сосед с возвращением без калипера.

In [ ]:
from dowhy import CausalModel

data_dw = d[['X1', 'X3', 'T', 'Y']].assign(X2sq=d.X2 ** 2, T=d['T'].astype(bool))
model = CausalModel(data=data_dw, treatment='T', outcome='Y', common_causes=['X1', 'X2sq', 'X3'])
estimand = model.identify_effect(proceed_when_unidentifiable=True)
psm_dw = model.estimate_effect(estimand, method_name='backdoor.propensity_score_matching', target_units='att')
ipw_dw = model.estimate_effect(estimand, method_name='backdoor.propensity_score_weighting', target_units='ate')
print(f'DoWhy PSM, ATT: {psm_dw.value:.3f} | наш PSM с калипером: {att_psm:.3f}')
print(f'DoWhy IPW, ATE: {ipw_dw.value:.3f} | наш Гаек: {ipw(d, e_hat["верная"])[1]:.3f}')

## Что унести из тетради

- **Сопоставление и взвешивание оценивают разное.** При неоднородном эффекте ATT и ATE расходятся, и сравнивать числа PSM и IPW как две оценки одного эффекта нельзя.
- **Гаек вместо Хорвица–Томпсона.** Нормировка на сумму весов гасит часть разброса от крупных весов.
- **Модель propensity score судят по балансу, а не по ROC-AUC.** Неверная модель выравнивает только те признаки, которые в нее вошли.
- **AIPW смещена, только когда неверны обе модели.** Но обе модели видят одни и те же измеренные признаки, и скрытый конфаундер AIPW не спасает.
- **Крайние веса — сигнал о слабом перекрытии.** Обрезка смещает оценку, trimming меняет вопрос; это решение пишут в отчет вместе с числом отброшенных объектов.

Источники:
1. [Rosenbaum & Rubin, 1983. The central role of the propensity score in observational studies for causal effects](https://doi.org/10.1093/biomet/70.1.41)
2. [Robins, Rotnitzky & Zhao, 1994. Estimation of Regression Coefficients When Some Regressors are not Always Observed](https://doi.org/10.1080/01621459.1994.10476818)
3. [Bang & Robins, 2005. Doubly Robust Estimation in Missing Data and Causal Inference Models](https://doi.org/10.1111/j.1541-0420.2005.00377.x)
4. [Austin, 2011. Optimal caliper widths for propensity-score matching when estimating differences in means and differences in proportions in observational studies](https://doi.org/10.1002/pst.433)
5. [Crump, Hotz, Imbens & Mitnik, 2009. Dealing with limited overlap in estimation of average treatment effects](https://doi.org/10.1093/biomet/asn055)